In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(os.path.join(path, "Q3_data.csv"))

In [ ]:
# Task 2: Write your code here:
print(df.head())

In [ ]:
# Task 3: Write your code here:
print(df.info())

In [ ]:
# Task 4: Write your code here:
print(df.describe())

In [ ]:
# Task 1: Write your code here:
df = df.fillna(df.mean(numeric_only=True))

In [ ]:
# Task 2: Write your code here:
df.drop_duplicates(inplace=True)

In [ ]:
# Task 3: Write your code here:
Encode categorical variables (CatBoost)

In [ ]:
# Task 4: Write your code here:
Apply feature scaling (StandardScaler)
scaler = StandardScaler()
features_to_scale = df.drop(columns=['target']).columns
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

In [ ]:
# Task 5: Write your code here:
Check for target imbalance
target_counts = df['target'].value_counts()
print("Target Balance:\n", target_counts)
is_imbalanced = target_counts.iloc[0] / target_counts.iloc[1] > 2
print(f"Is imbalanced? {is_imbalanced}")

In [ ]:
# Task 1: Write your code here:
Split the dataset into features (X) and target (y)
X = df.drop(columns=['target'])
y = df['target']

In [ ]:
# Task 2,3,4,5: Write your code here:
Print the averaged score across all folds
print(f"Average F1 Score across all folds: {np.mean(scores):.4f}")


from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
import numpy as np

# Task 2: Use StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Task 3: Train a CatBoostClassifier
scores = []

for train_index, val_index in skf.split(X, y):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]


    model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, silent=True)
    model.fit(X_train, y_train)


    preds = model.predict(X_val)
    score = f1_score(y_val, preds)
    scores.append(score)

# Task 5: Print the averaged score across all folds
print(f"Average F1 Score across all folds: {np.mean(scores):.4f}")



In [ ]:
# Task 1: Write your code here:
Plot feature importance
feature_importance = model.get_feature_importance()
sorted_idx = np.argsort(feature_importance)

plt.figure(figsize=(10, 8))
plt.barh(X.columns[sorted_idx], feature_importance[sorted_idx])
plt.xlabel("CatBoost Feature Importance")
plt.title("Feature Importance to find the Golden Feature")
plt.show()

In [ ]:
# Task 2: Write your code here:
Identify and print the 'golden feature'
golden_feature = X.columns[sorted_idx[-1]]
print(f"The Golden Feature is: {golden_feature}")

In [ ]:
# Task Bonus: Write your code here:
# Task 1: Create new X with only the golden feature
X_golden = df[[golden_feature]]

# Task 2: Run the same KFold loop with this single feature
golden_scores = []
for train_index, val_index in skf.split(X_golden, y):
    X_t, X_v = X_golden.iloc[train_index], X_golden.iloc[val_index]
    y_t, y_v = y.iloc[train_index], y.iloc[val_index]

    g_model = CatBoostClassifier(iterations=500, silent=True)
    g_model.fit(X_t, y_t)

    g_preds = g_model.predict(X_v)
    golden_scores.append(f1_score(y_v, g_preds))

# Task 3: Print and compare the accuracy
full_model_avg = np.mean(scores)
golden_model_avg = np.mean(golden_scores)

print(f"Full Model F1 Score: {full_model_avg:.4f}")
print(f"Golden Feature Only F1 Score: {golden_model_avg:.4f}")
print(f"Performance Retained: {(golden_model_avg/full_model_avg)*100:.2f}%")
